# 03. Huấn luyện QLoRA 4-bit cho LifeMate Voice-driven AI Agent
**Zalo AI Challenge 2026** — SFT Trajectory Fine-Tuning
- **Mô hình nền:** `Qwen/Qwen3-4B-Instruct-2507` (4B parameters)
- **Kỹ thuật:** QLoRA 4-bit (NF4) + SDPA Attention + Prompt Loss Masking (`labels = -100`)
- **Phần cứng:** Google Colab (GPU T4 16GB) hoặc Kaggle (GPU P100 / T4x2)
- **Thời gian huấn luyện:** ~25 - 30 phút cho 2 epochs trên 5.086 lượt huấn luyện
- **Bộ nhớ VRAM:** ~7.5 GB (thoải mái trong giới hạn GPU miễn phí)
- **Mục tiêu:** Nâng độ chính xác gọi tool, triệt tiêu lặp lệnh và bứt phá điểm số lên 75 - 85+!

## 1. Kiểm tra Card Đồ Họa (GPU Status)
Kiểm tra GPU T4 (Colab) hoặc P100/T4 (Kaggle).

In [ ]:
!nvidia-smi

## 2. Kết nối Google Drive (Nếu chạy trên Colab)
Lưu vĩnh viễn dữ liệu và trọng số LoRA sau khi train xong.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('✅ Đã kết nối Google Drive!')
except ImportError:
    print('ℹ️ Đang chạy trên Kaggle / Local machine, không cần mount Drive.')

## 3. Cài đặt các thư viện AI cần thiết

In [ ]:
!pip install -q "transformers>=4.45" accelerate peft bitsandbytes datasets sentencepiece

## 4. Cập nhật mã nguồn & Chuẩn bị dữ liệu Train (1.545 kịch bản)

In [ ]:
import os
import shutil
from pathlib import Path

# Tự động nhận diện Colab vs Kaggle
IS_COLAB = Path('/content').exists()
IS_KAGGLE = Path('/kaggle').exists()
WORKSPACE = Path('/content/LifeMateAgent') if IS_COLAB else (Path('/kaggle/working/LifeMateAgent') if IS_KAGGLE else Path('.'))
ADAPTER_OUT = '/content/drive/MyDrive/LifeMateAgent/adapters/qwen3_4b_lora' if IS_COLAB else '/kaggle/working/qwen3_4b_lora'
SUBMISSION_OUT = '/content/outputs/submission_public_test.jsonl' if IS_COLAB else '/kaggle/working/submission_public_test.jsonl'

# 1. Clone hoặc pull code mới nhất
if not WORKSPACE.exists():
    !git clone https://github.com/Platypus27-coder/zalo-ai-challenge.git {WORKSPACE}

%cd {WORKSPACE}
!git checkout main
!git pull origin main

# 2. Chuẩn bị dữ liệu train+public_test (Hỗ trợ cả Drive và Kaggle Input)
DATA_DIR = WORKSPACE / 'train+public_test'
if not (DATA_DIR / 'train' / 'train.jsonl').exists():
    # Tìm file zip trên Drive hoặc Kaggle Input
    all_zips = []
    if IS_KAGGLE and Path('/kaggle/input').exists():
        all_zips.extend(list(Path('/kaggle/input').rglob('*.zip')))
    all_zips.extend([
        Path('/content/drive/MyDrive/LifeMateAgent/data/train+public_test.zip'),
        Path('/content/drive/MyDrive/train+public_test.zip'),
        Path('/content/drive/MyDrive/LifeMateAgent/train+public_test.zip'),
    ])
    for pz in all_zips:
        if pz.exists() and 'train' in pz.name.lower():
            print(f'📦 Đang giải nén dữ liệu từ: {pz}')
            !unzip -q -o "{pz}" -d {WORKSPACE}/
            break

    # Nếu user upload Dataset lên Kaggle (Kaggle đã tự giải nén sẵn trong /kaggle/input)
    if not (DATA_DIR / 'train' / 'train.jsonl').exists() and IS_KAGGLE and Path('/kaggle/input').exists():
        found = list(Path('/kaggle/input').rglob('train.jsonl'))
        if found:
            train_folder = found[0].parent
            dataset_root = train_folder.parent
            print(f'📦 Tìm thấy dữ liệu Kaggle đã giải nén tại: {dataset_root}')
            DATA_DIR.mkdir(parents=True, exist_ok=True)
            !cp -r "{dataset_root}"/* {DATA_DIR}/ 2>/dev/null || cp -r "{train_folder}" {DATA_DIR}/train

if (DATA_DIR / 'train' / 'train.jsonl').exists():
    print('\n✅ Dữ liệu train (1.545 kịch bản) đã sẵn sàng!')
else:
    print('\n⚠️ Chưa tìm thấy train.jsonl! Hãy kiểm tra file zip trên Drive/Kaggle Input.')


## 5. Bóc tách 1.545 kịch bản thành 5.086 lượt huấn luyện ChatML
Script tự động mô phỏng Teacher-Forcing từng bước:
- Lượt 1: Yêu cầu người dùng -> Gọi tool mở đầu (tìm danh bạ, xem thời tiết,...)
- Lượt 2..N: Nhận kết quả từ môi trường -> Gọi tool hành động (chuyển tiền, nhắn tin,...)
- Lượt cuối: Xong việc -> Nói `DONE`.
- Lượt ngoài phạm vi: Câu nói chào hỏi/tán gẫu -> Nói `DONE`.

In [ ]:
# Sinh 5.086 lượt dữ liệu huấn luyện SFT trong 3 giây
!python scripts/prepare_sft_data.py

## 6. Bắt đầu Huấn Luyện QLoRA 4-bit (Thời gian: ~25 - 30 phút)
Quá trình huấn luyện sử dụng:
- **NF4 4-bit Base:** Giữ VRAM ở mức ~7.5 GB.
- **SDPA Attention:** Chống tràn bộ nhớ attention matrix.
- **Prompt Loss Masking:** Chỉ tính hàm mất mát trên câu trả lời của trợ lý, giúp hội tụ cực nhanh.
- **Cosine LR Scheduler:** 2e-4 với 5% warmup.

In [ ]:
!python scripts/train_lora.py \
    --epochs 3 \
    --lr 2e-4 \
    --batch-size 1 \
    --grad-accum 8 \
    --save-steps 200 \
    --resume \
    --output-dir {ADAPTER_OUT}


## 7. Kiểm tra kết quả & Tải LoRA Adapter về máy
Sau khi train xong, adapter LoRA (~50MB) đã được lưu vĩnh viễn trên Google Drive hoặc Kaggle Output.

In [ ]:
import shutil
from pathlib import Path

adapter_dir = Path(ADAPTER_OUT)
if (adapter_dir / 'adapter_model.safetensors').exists():
    print(f'🎉 Huấn luyện thành công 100%! Adapter lưu tại: {adapter_dir}')
    # Nếu trên Colab, tải thêm 1 bản zip về máy tính qua trình duyệt
    try:
        from google.colab import files
        zip_path = Path('/content/qwen3_4b_lora')
        shutil.make_archive(str(zip_path), 'zip', str(adapter_dir))
        print('⬇️ Đang tải adapter về máy tính của bạn...')
        files.download(str(zip_path) + '.zip')
    except Exception:
        pass
else:
    print('❌ Chưa tìm thấy adapter_model.safetensors!')

## 8. Đẩy LoRA Adapter lên Hugging Face Hub (Khuyên dùng - Private Repo)
Adapter LoRA chỉ nặng **~50MB**. Đưa lên Hugging Face Hub giúp lưu trữ vĩnh viễn và có thể nạp thẳng ở bất kỳ đâu.

### 🔑 Cách lấy Hugging Face Token miễn phí (30 giây):
1. Truy cập: https://huggingface.co/settings/tokens
2. Nhấp nút **Create new token** $\rightarrow$ Chọn Type: **Write** (bắt buộc chọn Write để có quyền tải model lên) $\rightarrow$ Đặt tên bất kỳ $\rightarrow$ Nhấp **Create token**.
3. Sao chép chuỗi mã bắt đầu bằng `hf_...` và dán vào biến `HF_TOKEN` ở ô code bên dưới.

In [ ]:
# ==============================================================================
# ĐIỀN THÔNG TIN TÀI KHOẢN HUGGING FACE CỦA BẠN TẠI ĐÂY:
# ==============================================================================
HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"   # <-- DÁN WRITE TOKEN CỦA BẠN VÀO ĐÂY
HF_REPO_ID = "your_username/lifemate-qwen3-4b-lora" # <-- ĐỔI 'your_username' THÀNH TÊN TÀI KHOẢN CỦA BẠN

token_arg = f"--token {HF_TOKEN}" if HF_TOKEN and not HF_TOKEN.startswith("hf_xxx") else ""

!python scripts/push_to_hub.py \
    --adapter-dir {ADAPTER_OUT} \
    --repo-id {HF_REPO_ID} \
    {token_arg}


## 9. Tạo bài nộp chính thức cho 634 mẫu Public Test
Chạy toàn bộ 634 mẫu Public Test với Não bộ LoRA mới và bộ lọc Guardrails chống lặp/chống ảo giác:
- Tự động đổi tên bài nộp cũ (nếu có) để lưu lại đối chứng và tránh Auto-Resume nhận nhầm 634 mẫu cũ.
- Cứ 25 mẫu tự động sao lưu checkpoint về Google Drive.

In [ ]:
import time, shutil
from pathlib import Path

# 1. Đổi tên file nộp cũ trên Drive (nếu có) để lưu làm mốc đối chiếu 10.27 điểm
drive_sub = Path('/content/drive/MyDrive/LifeMateAgent/submissions/submission_public_test.jsonl')
if drive_sub.exists():
    backup_sub = drive_sub.with_name(f'submission_public_test_old_{int(time.time())}.jsonl')
    shutil.move(str(drive_sub), str(backup_sub))
    print(f'📦 Đã đổi tên file bài nộp cũ trên Drive thành: {backup_sub.name}')

# 2. Xóa file đệm tạm trong session Colab (nếu có)
local_sub = Path('/content/outputs/submission_public_test.jsonl')
if local_sub.exists():
    local_sub.unlink()

!mkdir -p $(dirname {SUBMISSION_OUT})

# 3. Chạy inference chính thức 634 mẫu Public Test
# Có thể truyền đường dẫn adapter trên Drive ({ADAPTER_OUT}) hoặc trực tiếp từ Hugging Face ({HF_REPO_ID})!
!python scripts/generate_submission.py \
    --split public_test \
    --asr-mode precomputed \
    --agent-mode transformers \
    --model-name Qwen/Qwen3-4B-Instruct-2507 \
    --adapter-path {ADAPTER_OUT} \
    --output {SUBMISSION_OUT} \
    --verbose